[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/hubbard_dimer/hubbard_dimer.ipynb)

# The Hubbard Dimer

This example computes the energy spectrum, ground state, spin exchange, and charge transfer of the Hubbard dimer, a two-site model of interacting electrons introduced by Hubbard (1963). Two electrons hop between a left site and a right site while paying an on-site repulsion energy whenever both occupy the same site. Without repulsion the electrons delocalize across both sites; when sharing a site is costly, double occupancy gives way to brief virtual excursions that drive the slow spin exchange of Anderson (1959).

Each spin-orbital is modelled as an orthogonal vector, and a two-electron state is modelled as a bivector whose wedge product enforces Pauli exclusion and fermionic exchange signs. Any one-electron map extends to two-electron pairs through the incidence trace `(one_electron ^ (Pair & Complement)).trace(1, 3)`, which removes either occupied orbital via `Pair & Complement` and wedges its image back in with its exchange sign. Hopping couples the singlet only to the symmetric double occupancy, and on that two-state block the shifted map `hamiltonian - repulsion / 2 * Pair` squares to a scalar rate squared, giving closed-form block energies, the projected ground state, and real cosine and sine state parts in time.

In second-quantized Fock-space notation, two-electron basis states are built by ordered creation operators $\hat c^\dagger_{i\sigma}\hat c^\dagger_{j\sigma'}|0\rangle$ obeying $\{\hat c_{i\sigma}, \hat c^\dagger_{j\sigma'}\} = \delta_{ij}\delta_{\sigma\sigma'}$ and $\{\hat c_{i\sigma}, \hat c_{j\sigma'}\} = 0$, lifting a one-body matrix $h_{i\sigma, j\sigma'}$ sums $\sum_{i\sigma, j\sigma'} h_{i\sigma, j\sigma'}\,\hat c^\dagger_{i\sigma}\hat c_{j\sigma'}$, and on-site repulsion adds $U\sum_i \hat n_{i\uparrow}\hat n_{i\downarrow}$. Time evolution acts on complex vectors $|\psi(\tau)\rangle = e^{-i\hat H\tau}|\psi(0)\rangle$ in $\mathbb{C}^6$, and strong-repulsion exchange reduces to the Pauli-matrix tensor sum $J\,\mathbf S_L\cdot\mathbf S_R = \tfrac{J}{4}(\sigma_{Lx}\otimes\sigma_{Rx} + \sigma_{Ly}\otimes\sigma_{Ry} + \sigma_{Lz}\otimes\sigma_{Rz})$ with $J = 4t^2/U$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.quantum.hubbard_dimer import render
from examples.util.animation import save_figure, show_states
from numga import Algebra, NumpyContext, stack

np.set_printoptions(precision=4, suppress=True)

# Orbitals within each site are ordered spin up then spin down: (a, b) on the left and (c, d) on the right.
ga = Algebra("a+b+c+d+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                     # Scalar+
Orbital = ga.gatype.vector()                                    # Vector+
Pair = ga.gatype.bivector()                                     # Bivector+
Complement = ga.gatype.antivector()                             # Antivector-
Hopping = ga.gatype((Orbital, Orbital))
Hamiltonian = ga.gatype((Pair, Pair))

left = stack([mv.a, mv.b])                                      # [2] Vector+
right = stack([mv.c, mv.d])                                     # [2] Vector+
orbitals = stack([mv.a, mv.b, mv.c, mv.d])                      # [4] Vector+
configurations = stack([mv.ab, mv.ac, mv.ad, mv.bc, mv.bd, mv.cd])   # [6] Bivector+
doubles = stack([mv.ab, mv.cd])                                 # [2] Bivector+

## Hopping and energy spectrum

One-electron hopping moves an electron to the opposite site with its spin preserved and extends as a derivation acting on each occupied orbital of a pair in turn. Equal-spin pairs cannot hop into an already occupied orbital, and the opposite-spin triplet cancels under exchange, so all three triplets remain at zero energy. Hopping couples the singlet only to the symmetric double occupancy, while the antisymmetric double occupancy stays uncoupled at the repulsion energy itself. On the two-state singlet block, subtracting half the repulsion from the Hamiltonian leaves a traceless symmetric map whose square is a scalar rate squared, placing the block energies at half the repulsion plus or minus that rate and lowering the singlet below the triplets by the gap `rate - repulsion / 2`.

In matrix notation on the basis of the singlet and the symmetric double occupancy, the restricted Hamiltonian reads as $\begin{pmatrix}0 & -2t \\ -2t & U\end{pmatrix}$, with eigenvalues $U/2 \pm \sqrt{(U/2)^2 + 4t^2}$ and strong-repulsion singlet-triplet gap $4t^2/U + O(t^4/U^3)$.

In [ ]:
hopping = 1.0
names = ("Delocalized", "Crossover", "Localized spins")
repulsion_ratios = np.array([0, 4, 20])
repulsions = hopping * repulsion_ratios
n_samples = 241
sweep = np.linspace(0.0, repulsions[-1], n_samples)
# Strong-repulsion asymptote of the singlet-triplet gap, omitting the singularity at zero repulsion.
strong_repulsion = sweep[1:]
perturbative = 4 * hopping**2 / strong_repulsion

In [ ]:
def pair_extension(one_electron: Hopping) -> Hamiltonian:
    """An orbital map extends to two-electron pairs as a derivation acting on each wedge factor in turn.
    Joining `Pair & Complement` removes one occupied orbital, wedging `one_electron` inserts its image
    with the fermionic exchange sign, and the trace contracts the removed orbital with `Complement`."""
    return (one_electron ^ (Pair & Complement)).trace(1, 3)     # [] Bivector+ <- Bivector+


# Unit-strength hopping moves an electron to the opposite site with its spin preserved, then extends to pairs.
hop = (left * (right | Orbital) + right * (left | Orbital)).sum(axis=0)   # [] Vector+ <- Vector+
pair_hop = pair_extension(hop)                                  # [] Bivector+ <- Bivector+

singlet = (mv.ad - mv.bc) / np.sqrt(2)                          # [] Bivector+
zero_triplet = (mv.ad + mv.bc) / np.sqrt(2)                     # [] Bivector+
triplets = stack([mv.ac, zero_triplet, mv.bd])                  # [3] Bivector+
symmetric_double = (mv.ab + mv.cd) / np.sqrt(2)                 # [] Bivector+
# Each dyad projects onto one shared-site configuration; their sum counts double occupancy.
double_occupancy = (doubles * doubles.reverse().scalar_product(Pair)).sum(axis=0)   # [] Bivector+ <- Bivector+

In [ ]:
def hamiltonian(strength: float, repulsion: np.ndarray) -> Hamiltonian:
    """Two-electron hopping at `strength` competes with on-site `repulsion` (`[...] Bivector+ <- Bivector+`)."""
    return -strength * pair_hop + repulsion * double_occupancy  # [...] Bivector+ <- Bivector+


def block_rate(strength: float, repulsion: np.ndarray) -> np.ndarray:
    """Applying the Hamiltonian minus half `repulsion` twice on the singlet block multiplies by the
    square of this rate (`[...]`)."""
    return np.sqrt((repulsion / 2) ** 2 + 4 * strength**2)


energies = hamiltonian(hopping, sweep).eigvalsh()               # [n_samples, 6] Scalar+
exchange_gap = block_rate(hopping, sweep) - sweep / 2           # [n_samples]
save_figure(render.spectrum_figure(sweep, hopping, energies, exchange_gap, perturbative, strong_repulsion), "hubbard_dimer_spectrum");

In the left panel, the purple singlet ground state curves toward zero from below as repulsion grows, the three blue triplets stay flat at zero energy, and the two red excited singlets rise linearly with repulsion as shared-site states. In the right panel, the singlet-triplet gap falls from twice the hopping strength at zero repulsion onto the dashed strong-repulsion asymptote.

## Ground-state configurations and spin alignment

On the two-state singlet block, subtracting the shifted Hamiltonian from `block_rate` annihilates the upper eigenstate and multiplies the lower eigenstate by twice `block_rate`, projecting the singlet directly onto the ground state before normalization. Without repulsion the ground state places both electrons on a single site half the time. Strong repulsion suppresses double occupancy and leaves one electron per site in a spin singlet with spin alignment `-3/4`, whereas each shared-site configuration carries zero local spin and dilutes that antiparallel alignment.

In bra-ket notation, the singly occupied singlet reads as $(|\uparrow,\downarrow\rangle - |\downarrow,\uparrow\rangle)/\sqrt{2}$, and the inter-site spin alignment operator reads as $\mathbf S_L\cdot\mathbf S_R$ in units of $\hbar^2$, taking eigenvalue $-3/4$ on the singlet, $+1/4$ on the three triplets, and $0$ on doubly occupied configurations.

In [ ]:
singlet_weight = singlet * singlet.reverse().scalar_product(Pair)   # [] Bivector+ <- Bivector+
# Doubly occupied states carry zero local spin, while singlets have spin alignment -3/4 and triplets +1/4.
spin_alignment = (Pair - double_occupancy - 4 * singlet_weight) / 4   # [] Bivector+ <- Bivector+


def ground_state(strength: float, repulsion: np.ndarray) -> Pair:
    """Subtracting the offset Hamiltonian from `block_rate` projects the singlet onto the normalized
    ground state (`[...] Bivector+`)."""
    offset_singlet = hamiltonian(strength, repulsion)(singlet) - repulsion / 2 * singlet   # [...] Bivector+
    lower = block_rate(strength, repulsion) * singlet - offset_singlet   # [...] Bivector+
    return lower / lower.scalar_norm_squared().square_root()    # [...] Bivector+


def expectation(state: Pair, observable: Hamiltonian) -> Scalar:
    """Pairing `state` with `observable(state)` gives the quadratic expectation value (`[...] Scalar+`)."""
    return state.reverse().scalar_product(observable(state))    # [...] Scalar+


grounds = ground_state(hopping, repulsions)                     # [3] Bivector+
ground_energies = expectation(grounds, hamiltonian(hopping, repulsions))   # [3] Scalar+
ground_doubles = expectation(grounds, double_occupancy)         # [3] Scalar+
ground_alignment = expectation(grounds, spin_alignment)         # [3] Scalar+

In [ ]:
def probabilities(state: Pair) -> Scalar:
    """Squared overlaps with the six basis configurations give the occupation probabilities
    (`[..., 6] Scalar+`) of a real pair `state`."""
    return configurations.reverse().scalar_product(state[..., None]).squared()   # [..., 6] Scalar+


ground_probabilities = probabilities(grounds)                   # [3, 6] Scalar+
save_figure(render.ground_states_figure(names, repulsion_ratios, ground_probabilities), "hubbard_dimer_ground_states");

In the delocalized row at zero repulsion, weight spreads equally across the two shared-site columns and the two opposite-spin singly occupied columns, while both equal-spin columns stay dark. As repulsion increases down the rows, weight concentrates into the two opposite-spin singly occupied columns and the shared-site columns fade out.

In [ ]:
print("ground energies:", ground_energies.to_array())
print("double occupancy per regime:", ground_doubles.to_array())
print("spin alignment per regime:", ground_alignment.to_array())

## Spin exchange across repulsion regimes

Starting from spin up on the left site and spin down on the right site decomposes the initial pair equally into the zero-spin triplet and the singlet. Time evolution splits any real initial state into real cosine and sine state parts obtained by acting with the cosine and sine of the Hamiltonian times time, so each basis configuration's probability is the sum of its squared overlaps with those two real bivectors. The triplet stays fixed at zero energy inside the cosine part, while on the singlet block the shifted Hamiltonian acts at `block_rate` and half the repulsion mixes the cosine and sine parts. Scaling time by each regime's own singlet-triplet gap aligns one full spin-exchange cycle across weak, intermediate, and strong repulsion.

In bra-ket notation, the cosine and sine parts correspond to $\cos(\hat H\tau)|\psi(0)\rangle$ and $-\sin(\hat H\tau)|\psi(0)\rangle$ inside $e^{-i\hat H\tau}|\psi(0)\rangle = \cos(\hat H\tau)|\psi(0)\rangle - i\sin(\hat H\tau)|\psi(0)\rangle$, the spin-swap probability reads as $|\langle\downarrow,\uparrow|e^{-i\hat H\tau}|\uparrow,\downarrow\rangle|^2$, and the localized Heisenberg model predicts $\sin^2(J\tau/2)$.

In [ ]:
n_steps = 1200
n_cycles = 1
phase = np.linspace(0.0, n_cycles * 2 * np.pi, n_steps + 1)
spin_only = np.sin(phase / 2) ** 2

In [ ]:
def exchange_parts(strength: float, repulsion: np.ndarray, times: np.ndarray) -> tuple[Pair, Pair]:
    """Starting from spin up on the left and spin down on the right, time evolution over `times` yields
    the cosine and sine state parts (`[...] Bivector+`, `[...] Bivector+`). The zero-energy triplet stays
    in the cosine part while the singlet rotates at `block_rate` and mixes under half `repulsion`."""
    rate = block_rate(strength, repulsion)
    offset_singlet = hamiltonian(strength, repulsion)(singlet) - repulsion / 2 * singlet   # [...] Bivector+
    singlet_cosine = np.cos(rate * times) * singlet             # [...] Bivector+
    singlet_sine = np.sin(rate * times) / rate * offset_singlet   # [...] Bivector+
    half_phase = repulsion * times / 2
    cosine_part = (zero_triplet + np.cos(half_phase) * singlet_cosine - np.sin(half_phase) * singlet_sine) / np.sqrt(2)   # [...] Bivector+
    sine_part = (np.sin(half_phase) * singlet_cosine + np.cos(half_phase) * singlet_sine) / np.sqrt(2)   # [...] Bivector+
    return cosine_part, sine_part


gaps = block_rate(hopping, repulsions) - repulsions / 2         # [3]
times = phase / gaps[:, None]                                   # [3, n_steps + 1]
cosine_part, sine_part = exchange_parts(hopping, repulsions[:, None], times)   # [3, n_steps + 1] Bivector+, Bivector+
exchange_probabilities = probabilities(cosine_part) + probabilities(sine_part)   # [3, n_steps + 1, 6] Scalar+
save_figure(render.exchange_figure(phase, exchange_probabilities, spin_only), "hubbard_dimer_exchange");

Across the three panels from delocalized to localized spins, the purple swapped-spin curve approaches the dashed spin-only sine squared while the red double-occupancy curve drops toward zero. Free hopping swaps the spins through large-amplitude charge motion, intermediate repulsion superimposes rapid shared-site oscillations on the swap, and strong repulsion tracks the localized-spin curve with only brief virtual double occupancy.

In [ ]:
print("peak double occupancy per regime:", (exchange_probabilities[..., 0] + exchange_probabilities[..., 5]).max(axis=-1).to_array())
print("peak swap deviation from spin-only:", (exchange_probabilities[..., 3] - spin_only).abs().max(axis=-1).to_array())

## Configuration amplitudes on planar dials

For each reachable basis configuration, pairing the cosine and sine state parts with that configuration defines a point on a planar dial whose squared distance from the centre equals the configuration probability. Splitting out the zero-spin triplet projection decomposes each singly occupied amplitude into a stationary triplet vector and a turning singlet-block vector added tip to tail, so spin exchange appears directly as destructive and constructive interference between the two vectors.

In [ ]:
dial_stride = 6
frame_duration_ms = 40

In [ ]:
def exchange_states(strength: float, repulsion: np.ndarray, exchange_phase: np.ndarray) -> Iterator[tuple[Pair, Pair]]:
    """Yields `(cosine_part, sine_part)` (`[...] Bivector+`, `[...] Bivector+`) step by step along
    `exchange_phase`, with each regime scaled to its own singlet-triplet gap."""
    gap = block_rate(strength, repulsion) - repulsion / 2       # [...]
    for step_phase in exchange_phase:
        yield exchange_parts(strength, repulsion, step_phase / gap)


dial_rows = [f"{name}\nrepulsion {ratio:g}" for name, ratio in zip(names, repulsion_ratios)]
dial_history = show_states(
    exchange_states(hopping, repulsions, phase[::dial_stride]),
    render.dials_animation, dial_rows, configurations, zero_triplet,
    name="hubbard_dimer_dials", duration_ms=frame_duration_ms,
)

On the two singly occupied dials in the middle columns, the purple singlet-block segment turns around the tip of the stationary grey triplet segment, cancelling on the initial configuration when the two segments add on the swapped configuration. Without repulsion the purple segment swings linearly across the dial, in the crossover row it traces rosettes from rapid charge motion, and at strong repulsion it circles the grey tip cleanly while the outer shared-site dials barely move.

## Site-dependent magnetic field

A magnetic field opposite on the two sites, as in adjacent semiconductor quantum dots under a field gradient, raises spin up and lowers spin down on the left site while reversing those shifts on the right site. Extending that one-electron map to pairs via `pair_extension` leaves the opposite-spin product configurations as eigenstates with opposite energies, mixing the singlet with the zero-spin triplet so the two-state block no longer closes on itself. Diagonalizing the full pair Hamiltonian instead decomposes the initial pair across all six eigenmodes, and each eigenmode turns its share through the cosine and sine of its eigenenergy times time.

In Pauli-matrix notation, the site-dependent Zeeman term reads as $\tfrac{\Delta}{2}(\sigma_{Lz} - \sigma_{Rz})$, and at strong repulsion the singlet and zero-spin triplet form the two-level subspace studied in the [two spins](../two_spins/two_spins.ipynb) notebook, precessing about an axis tilted between the exchange splitting and the field difference.

In [ ]:
field_ratios = np.array([0, 1, 5])

In [ ]:
# A magnetic field opposite on the two sites raises spin up and lowers spin down on the left, and reverses on the right.
field_signs = np.array([1, -1, -1, 1])
field_difference = pair_extension((field_signs * (orbitals * (orbitals | Orbital))).sum(axis=0) / 2)   # [] Bivector+ <- Bivector+


def mode_evolution(energies: Scalar, modes: Pair, start: Pair, times: np.ndarray) -> Iterator[tuple[Pair, Pair]]:
    """Yields `(cosine_part, sine_part)` (`[...] Bivector+`, `[...] Bivector+`) along `times` as each
    eigenmode in `modes` (`[..., 6] Bivector+`) turns its share of `start` at its energy in `energies`."""
    shares = modes * modes.reverse().scalar_product(start)      # [..., 6] Bivector+
    for step_time in times:
        angles = energies * step_time                           # [..., 6] Scalar+
        cosine_part = (shares * angles.cos()).sum(axis=-1)      # [...] Bivector+
        sine_part = (shares * angles.sin()).sum(axis=-1)        # [...] Bivector+
        yield cosine_part, sine_part


strongest_repulsion, strongest_gap = repulsions[-1], gaps[-1]
field_hamiltonians = hamiltonian(hopping, strongest_repulsion) + field_ratios * strongest_gap * field_difference   # [3] Bivector+ <- Bivector+
field_energies, field_modes = field_hamiltonians.eigh()         # [3, 6] Scalar+, Bivector+
field_rows = [f"field difference\n{ratio:g} exchange" for ratio in field_ratios]
field_history = show_states(
    mode_evolution(field_energies, field_modes, mv.ad, phase[::dial_stride] / strongest_gap),
    render.dials_animation, field_rows, configurations, zero_triplet,
    name="hubbard_dimer_field_dials", duration_ms=frame_duration_ms,
)

At zero field difference in the top row, the grey triplet segment stays fixed while the purple segment completes a full spin swap. When the field difference matches the exchange gap in the middle row, the grey triplet segment turns as well and only part of the weight reaches the swapped column. At five times the exchange gap in the bottom row, the two segments counter-rotate on the swapped dial and keep nearly all probability in the initial spin configuration.

In [ ]:
field_cosine = stack([state[0] for state in field_history], axis=1)   # [3, ...] Bivector+
field_sine = stack([state[1] for state in field_history], axis=1)     # [3, ...] Bivector+
field_probabilities = probabilities(field_cosine) + probabilities(field_sine)   # [3, ..., 6] Scalar+
print("peak swapped-spin probability per field difference:", field_probabilities[..., 3].max(axis=-1).to_array())

## Site asymmetry and charge transfer

Lowering the left site energy and raising the right site energy equally for both spins draws charge toward the lower site. Extending that site-offset map to pairs via `pair_extension` produces `site_difference`, which enters the Hamiltonian scaled by the asymmetry and simultaneously serves as the observable measuring half the occupation of the right site minus the left site. On-site repulsion and site asymmetry compete directly in the ground state, holding one electron on each site until the site offset overcomes the cost of double occupancy; the [Hubbard ring](../hubbard_ring/hubbard_ring.ipynb) notebook extends the same `pair_extension` trace to four electrons on four sites, and [Carrascal, Ferrer, Smith, and Burke (2015)](https://arxiv.org/abs/1502.02194) build a density-functional benchmark around this asymmetric dimer.

In second-quantized notation, the site-asymmetry term reads as $\tfrac{\Delta v}{2}\sum_\sigma(\hat n_{R\sigma} - \hat n_{L\sigma})$, and minus twice its ground-state expectation gives the left-minus-right occupation difference $\langle \hat n_L - \hat n_R \rangle$.

In [ ]:
n_asymmetries = 401
asymmetries = hopping * np.linspace(0.0, 2 * repulsions[-1], n_asymmetries)

In [ ]:
# Site offsets of -1/2 on the left and +1/2 on the right measure half the right occupation minus the left.
site_signs = np.array([-1, -1, 1, 1])
site_difference = pair_extension((site_signs * (orbitals * (orbitals | Orbital))).sum(axis=0) / 2)   # [] Bivector+ <- Bivector+

asymmetric_hamiltonians = hamiltonian(hopping, repulsions[:, None]) + asymmetries * site_difference   # [3, n_asymmetries] Bivector+ <- Bivector+
_, asymmetric_modes = asymmetric_hamiltonians.eigh()            # [3, n_asymmetries, 6] Bivector+
asymmetric_grounds = asymmetric_modes[..., 0]                   # [3, n_asymmetries] Bivector+
charge_differences = -2 * expectation(asymmetric_grounds, site_difference)   # [3, n_asymmetries] Scalar+
save_figure(render.transfer_figure(asymmetries / hopping, charge_differences, names, repulsion_ratios), "hubbard_dimer_transfer");

Without repulsion, the purple curve rises immediately toward an occupation difference of two electrons on the lower site. Intermediate repulsion delays that rise in the blue curve until the asymmetry reaches four times hopping, while strong repulsion holds the red curve near zero across a broad Mott plateau before stepping sharply toward two around twenty.

In [ ]:
half_index, twice_index = np.searchsorted(asymmetries, [0.5 * repulsions[-1], 2 * repulsions[-1]])
print("occupation difference at half the strongest repulsion:", charge_differences[:, half_index].to_array())
print("occupation difference at twice the strongest repulsion:", charge_differences[:, twice_index].to_array())

In [ ]:
# checks
# Equal-spin hops vanish, the opposite-spin triplet cancels, and the singlet maps to twice the symmetric double occupancy.
assert (pair_hop(triplets).scalar_norm_squared().square_root() < 1e-10).all()
assert ((pair_hop(singlet) - 2 * symmetric_double).scalar_norm_squared().square_root() < 1e-10).all()

# Applying the shifted Hamiltonian twice on the singlet block multiplies by the block rate squared,
# and the analytic gap and ground-state energy match the full spectrum.
sweep_hamiltonians = hamiltonian(hopping, sweep[:, None])       # [n_samples, 1] Bivector+ <- Bivector+
shift = sweep[:, None] / 2                                      # [n_samples, 1]
rate = block_rate(hopping, sweep[:, None])                      # [n_samples, 1]
block = stack([singlet, symmetric_double])                      # [2] Bivector+
block_step = sweep_hamiltonians(block) - shift * block          # [n_samples, 2] Bivector+
assert ((sweep_hamiltonians(block_step) - shift * block_step - rate**2 * block).scalar_norm_squared().square_root() < 1e-10).all()
assert (((energies[:, 1] - energies[:, 0]) - exchange_gap).abs() < 1e-10).all()
assert ((ground_energies - (repulsions / 2 - block_rate(hopping, repulsions))).abs() < 1e-10).all()

# Spin up on the left and spin down on the right splits equally into singlet and triplet, and stays normalized.
assert (((singlet + zero_triplet) / np.sqrt(2) - mv.ad).scalar_norm_squared().square_root() < 1e-10).all()
assert ((exchange_probabilities.sum(axis=-1) - 1).abs() < 1e-10).all()
dial_cosine = stack([state[0] for state in dial_history], axis=1)   # [3, ...] Bivector+
dial_sine = stack([state[1] for state in dial_history], axis=1)     # [3, ...] Bivector+
assert (((probabilities(dial_cosine) + probabilities(dial_sine)).sum(axis=-1) - 1).abs() < 1e-10).all()

# Opposite site fields keep up-down as an eigenstate, a strong field freezes the swap, and strong repulsion
# holds one electron per site until the site asymmetry outweighs it.
assert ((field_difference(mv.ad) - mv.ad).scalar_norm_squared().square_root() < 1e-10).all()
assert ((field_probabilities.sum(axis=-1) - 1).abs() < 1e-10).all()
assert (field_probabilities[-1, :, 3] < 0.05).all()
assert bool(charge_differences[-1, half_index] < 0.1)
assert bool(charge_differences[-1, twice_index] > 1.9)